# Intervention Verbs

Exercises the top-level do/replay/rerun entry points at smoke level and records expected failure shapes for invalid sites.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.ParamLog.shape",
    "tl.ParamLog.trainable",
    "tl.ParamLog.used_by_layers",
    "tl.accessors",
    "tl.accessors.GradFnAccessor",
    "tl.accessors.LayerAccessor",
    "tl.accessors.ModuleAccessor",
    "tl.batched_extract",
    "tl.bridge",
    "tl.bridge.brain_score",
    "tl.bridge.captum",
    "tl.bridge.depyf",
    "tl.bridge.dialz",
    "tl.bridge.gradcam",
    "tl.bridge.huggingface",
    "tl.bridge.inseq",
    "tl.bridge.lit",
    "tl.bridge.nnsight",
    "tl.bridge.profiler",
    "tl.bridge.repeng",
    "tl.bridge.rsatoolbox",
    "tl.bridge.sae_lens",
    "tl.bridge.shap",
    "tl.bridge.steering_vectors",
    "tl.bundle",
    "tl.bwd_hook",
    "tl.callbacks",
    "tl.callbacks.lightning",
    "tl.capture",
    "tl.capture.arg_positions",
    "tl.capture.flops",
    "tl.capture.output_tensors",
    "tl.capture.salient_args",
    "tl.capture.source_tensors",
    "tl.capture.tensor_tracking",
    "tl.clamp",
    "tl.compat",
    "tl.compat.CompatReport",
    "tl.compat.CompatRow",
    "tl.compat.Extractor",
    "tl.compat.from_fx",
    "tl.compat.from_huggingface",
    "tl.compat.from_ilg",
    "tl.compat.from_sentence_transformers",
    "tl.compat.from_timm",
    "tl.compat.from_torchextractor",
    "tl.compat.lovely",
    "tl.compat.report",
    "tl.compat.torchextractor",
    "tl.compat.torchshow",
    "tl.constants",
    "tl.constants.BUFFER_LOG_FIELD_ORDER",
    "tl.constants.FUNC_CALL_LOCATION_FIELD_ORDER",
    "tl.constants.GRAD_FN_LOG_FIELD_ORDER",
    "tl.constants.GRAD_FN_PASS_LOG_FIELD_ORDER",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")